In [10]:
# ==========================================
# 1. Importación de librerías y datos
# ==========================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
import itertools

# Cargar el dataset
df = pd.read_csv('../Data/video_game_reviews.csv')

#  OPTIMIZACIÓN: Usar una muestra del 20% de los datos
# SVM es O(n²) a O(n³) - extremadamente lento en datasets grandes
df = df.sample(n=10000, random_state=42)  # 10k samples en lugar de 47k
print(f"Usando muestra de {len(df)} registros para acelerar SVM")

# Variable objetivo
target = 'User Rating'

# Convertir variables categóricas antes de separar
cat_cols = df.select_dtypes(include=['object', 'category']).columns
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Ahora sí, separar features y target
features = [col for col in df.columns if col != target]

X = df[features]
y = df[target]

# División 60/20/20
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Escalado
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Datos listos para entrenar (numéricas + codificadas)")
print(f"Dimensiones finales: {X_train.shape}")


Usando muestra de 10000 registros para acelerar SVM
Datos listos para entrenar (numéricas + codificadas)
Dimensiones finales: (6000, 99)


In [11]:
# ==========================================
# 2. Definición de hiperparámetros (3×3×3)
# ==========================================

# Requisito: 3 hiperparámetros con 3 valores cada uno
# Total: 3×3×3 = 27 combinaciones

# Hiperparámetro 1: C (regularización)
C_values = [0.1, 1, 10]

# Hiperparámetro 2: kernel (tipo de kernel)
kernel_values = ['linear', 'rbf', 'poly']  # 3 kernels diferentes

# Hiperparámetro 3: epsilon (margen de error tolerado)
epsilon_values = [0.01, 0.1, 1]

# Lista para guardar resultados
results = []

# Combinaciones de hiperparámetros
combinations = list(itertools.product(C_values, kernel_values, epsilon_values))

print(f"Total de combinaciones: {len(combinations)}")
print(f"Hiperparámetros a probar:")
print(f"  - C (regularización): {C_values}")
print(f"  - kernel (tipo): {kernel_values}")
print(f"  - epsilon (margen): {epsilon_values}")
print(f"\nNota: Usando muestra de 10k datos para mantener tiempo razonable (~15-20 min)")


Total de combinaciones: 27
Hiperparámetros a probar:
  - C (regularización): [0.1, 1, 10]
  - kernel (tipo): ['linear', 'rbf', 'poly']
  - epsilon (margen): [0.01, 0.1, 1]

Nota: Usando muestra de 10k datos para mantener tiempo razonable (~15-20 min)


In [12]:
# ==========================================
# 3. Entrenamiento con bucles anidados
# ==========================================

import time

print(f"Iniciando entrenamiento de {len(combinations)} modelos SVM...\n")
print(f"Esto puede tomar 15-20 minutos. Por favor espere...\n")
print(f"{'='*70}\n")

start_time = time.time()
model_count = 0

# Bucles anidados para cambiar hiperparámetros (según requisito b)
for C in C_values:
    for kernel in kernel_values:
        for epsilon in epsilon_values:
            model_count += 1
            print(f"[{model_count}/{len(combinations)}] C={C}, kernel={kernel}, epsilon={epsilon}")
            
            model_start = time.time()
            
            # Entrenar modelo SVM
            model = SVR(C=C, kernel=kernel, epsilon=epsilon, max_iter=1000)
            model.fit(X_train, y_train)
            
            model_time = time.time() - model_start

            # Predicciones en train y validación
            y_train_pred = model.predict(X_train)
            y_val_pred = model.predict(X_val)

            # Métricas (función de costo - MSE)
            train_mse = mean_squared_error(y_train, y_train_pred)
            val_mse = mean_squared_error(y_val, y_val_pred)

            # Guardar resultados
            results.append({
                'C': C,
                'Kernel': kernel,
                'Epsilon': epsilon,
                'Train MSE': train_mse,
                'Validation MSE': val_mse,
                'Time (s)': model_time
            })
            
            print(f"   Train MSE: {train_mse:.4f} | Val MSE: {val_mse:.4f} | Tiempo: {model_time:.1f}s\n")

total_time = time.time() - start_time
print(f"\n{'='*70}")
print(f" Entrenamiento finalizado en {total_time/60:.1f} minutos")
print(f"{'='*70}")


Iniciando entrenamiento de 27 modelos SVM...

Esto puede tomar 15-20 minutos. Por favor espere...


[1/27] C=0.1, kernel=linear, epsilon=0.01


c:\Users\marti\OneDrive\Documents\Diego\8VO SEMESTRE\INTRO A IA\PROYECTO 4\proyecto4_IA\proyectoIAEnv\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


   Train MSE: 1.8122 | Val MSE: 1.8315 | Tiempo: 1.8s

[2/27] C=0.1, kernel=linear, epsilon=0.1


c:\Users\marti\OneDrive\Documents\Diego\8VO SEMESTRE\INTRO A IA\PROYECTO 4\proyecto4_IA\proyectoIAEnv\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


   Train MSE: 1.6353 | Val MSE: 1.5915 | Tiempo: 3.1s

[3/27] C=0.1, kernel=linear, epsilon=1


...

   Train MSE: 2.7629 | Val MSE: 8.4410 | Tiempo: 5.5s

[26/27] C=10, kernel=poly, epsilon=0.1


c:\Users\marti\OneDrive\Documents\Diego\8VO SEMESTRE\INTRO A IA\PROYECTO 4\proyecto4_IA\proyectoIAEnv\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


   Train MSE: 2.7724 | Val MSE: 8.6648 | Tiempo: 5.1s

[27/27] C=10, kernel=poly, epsilon=1


c:\Users\marti\OneDrive\Documents\Diego\8VO SEMESTRE\INTRO A IA\PROYECTO 4\proyecto4_IA\proyectoIAEnv\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


   Train MSE: 3.3642 | Val MSE: 10.5828 | Tiempo: 13.7s


 Entrenamiento finalizado en 10.9 minutos


In [13]:
# ==========================================
# 4. Tabla comparativa de resultados
# ==========================================

# Crear DataFrame con resultados
results_df = pd.DataFrame(results)

# Ordenar por Validation MSE (mejor modelo arriba)
results_df_sorted = results_df.sort_values(by='Validation MSE')
results_df_sorted = results_df_sorted.reset_index(drop=True)

print("TABLA COMPARATIVA DE TODOS LOS MODELOS")
print("Columnas: Hiperparámetros | Error Train | Error Validación\n")
display(results_df_sorted)

# Identificar mejores hiperparámetros
best_params = results_df_sorted.iloc[0]
print("\n" + "="*70)
print(" MEJORES HIPERPARÁMETROS ENCONTRADOS:")
print("="*70)
print(f"C (regularización): {best_params['C']}")
print(f"Kernel (tipo): {best_params['Kernel']}")
print(f"Epsilon (margen): {best_params['Epsilon']}")
print(f"\nError de Entrenamiento (MSE): {best_params['Train MSE']:.4f}")
print(f"Error de Validación (MSE): {best_params['Validation MSE']:.4f}")
print("="*70)


TABLA COMPARATIVA DE TODOS LOS MODELOS
Columnas: Hiperparámetros | Error Train | Error Validación



,C,Kernel,Epsilon,Train MSE,Validation MSE,Time (s)
0,0.1,linear,1.00,1.473571,1.489636,4.061223
1,1.0,linear,1.00,1.507683,1.512889,8.444051
2,10.0,linear,1.00,1.507683,1.512889,2.933771
3,0.1,linear,0.10,1.635275,1.591452,3.112527
4,0.1,linear,0.01,1.812166,1.831547,1.840512
5,1.0,linear,0.01,1.805121,1.832486,10.625928
6,10.0,linear,0.01,1.805121,1.832486,4.518782
7,1.0,linear,0.10,2.201225,2.075654,9.511699
8,10.0,linear,0.10,2.201225,2.075654,4.486231
9,10.0,rbf,0.01,0.855439,2.417636,6.251799



 MEJORES HIPERPARÁMETROS ENCONTRADOS:
C (regularización): 0.1
Kernel (tipo): linear
Epsilon (margen): 1.0

Error de Entrenamiento (MSE): 1.4736
Error de Validación (MSE): 1.4896


In [14]:
# ==========================================
# 5. Error de Test y Predicción de Dato Nuevo
# ==========================================

print("EVALUACIÓN FINAL DEL MEJOR MODELO\n")

# Entrenar modelo final con mejores parámetros
best_model = SVR(
    C=best_params['C'],
    kernel=best_params['Kernel'],
    epsilon=best_params['Epsilon'],
    max_iter=1000
)
best_model.fit(X_train, y_train)

# d) Error de test
y_test_pred = best_model.predict(X_test)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_test_pred)

print(" MÉTRICAS EN CONJUNTO DE TEST:")
print(f"  - MSE (Mean Squared Error): {test_mse:.4f}")
print(f"  - RMSE (Root Mean Squared Error): {test_rmse:.4f}")
print(f"  - R² Score: {test_r2:.4f}")

# e) Predicción de dato nuevo (inventado)
print("\n" + "="*70)
print(" PREDICCIÓN PARA UN NUEVO JUEGO (dato inventado)")
print("="*70)

# Crear dato nuevo: características promedio del dataset
new_data = np.mean(X_train, axis=0).reshape(1, -1)
new_pred = best_model.predict(new_data)

print(f"\nCaracterísticas del juego: Valores promedio del dataset")
print(f"Predicción de User Rating: {new_pred[0]:.2f}")

# Comparar con promedio real
print(f"\nRating promedio real en dataset: {y_train.mean():.2f}")
print(f"Diferencia: {abs(new_pred[0] - y_train.mean()):.2f}")


EVALUACIÓN FINAL DEL MEJOR MODELO



c:\Users\marti\OneDrive\Documents\Diego\8VO SEMESTRE\INTRO A IA\PROYECTO 4\proyecto4_IA\proyectoIAEnv\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


 MÉTRICAS EN CONJUNTO DE TEST:
  - MSE (Mean Squared Error): 1.4925
  - RMSE (Root Mean Squared Error): 1.2217
  - R² Score: 0.9745

 PREDICCIÓN PARA UN NUEVO JUEGO (dato inventado)

Características del juego: Valores promedio del dataset
Predicción de User Rating: 29.62

Rating promedio real en dataset: 29.67
Diferencia: 0.05


## CONCLUSIONES DEL EXPERIMENTO


 PROCEDIMIENTO REALIZADO:

1. PREPARACIÓN DE DATOS:
   - Dataset original: 47,774 registros
   - Muestra utilizada: 10,000 registros (para viabilidad computacional)
   - Variables categóricas codificadas con One-Hot Encoding
   - Datos escalados con StandardScaler
   - División: 60% train, 20% validación, 20% test

2. HIPERPARÁMETROS EVALUADOS (3×3×3 = 27 combinaciones):
   a) C (regularización): [0.1, 1, 10]
      - Controla el trade-off entre error y complejidad del modelo
   
   b) Kernel (tipo de función): ['linear', 'rbf', 'poly']
      - Define cómo se mapean los datos en el espacio de características
   
   c) Epsilon (margen de error): [0.01, 0.1, 1]
      - Define el tubo de tolerancia en la regresión

3. ENTRENAMIENTO:
   - Se usaron bucles anidados (for dentro de for) para probar todas las combinaciones
   - Cada modelo se evaluó en train y validación
   - Se midió el tiempo de entrenamiento de cada configuración

4. EVALUACIÓN:
   - Métrica principal: MSE (Mean Squared Error)
   - Se seleccionó el modelo con menor error en validación
   - Evaluación final en conjunto de test (datos no vistos)



 ANÁLISIS DE RESULTADOS:

1. MEJOR CONFIGURACIÓN:
   - Los hiperparámetros óptimos se muestran arriba
   - El modelo con mejor validación generalizó bien al test

2. OBSERVACIONES SOBRE HIPERPARÁMETROS:
   
   • C (Regularización):
     - Valores bajos (0.1): Modelo más simple, menor overfitting
     - Valores altos (10): Modelo más complejo, puede sobreajustar
   
   • Kernel:
     - Linear: Más rápido, mejor para relaciones lineales
     - RBF: Más flexible, captura relaciones no lineales
     - Poly: Muy lento, útil para patrones polinómicos
   
   • Epsilon:
     - Valores pequeños: Modelo más estricto
     - Valores grandes: Más tolerante a errores

3. RENDIMIENTO DEL MODELO:
   - El MSE de test indica qué tan bien predice en datos nuevos
   - R² muestra el porcentaje de varianza explicada
   - La predicción del dato inventado valida el comportamiento del modelo



 RECOMENDACIONES PARA MEJORAR:

1. Usar el dataset completo (47k registros) si se tiene suficiente tiempo/recursos
2. Probar más valores de hiperparámetros (grid más fino)
3. Implementar validación cruzada (cross-validation) para mayor robustez
4. Aplicar selección de características para reducir dimensionalidad
5. Comparar SVM con otros algoritmos (Random Forest, XGBoost, etc.)
6. Analizar qué características son más importantes para la predicción



  LIMITACIONES:

- SVM es O(n²) a O(n³): muy lento en datasets grandes
- Se usó muestra reducida (10k) para mantener tiempo de ejecución razonable
- El kernel polynomial es extremadamente lento pero se incluyó para completitud

